# Credit Risk Prediction - Give Me Some Credit

Attach the competition data in Kaggle and run the code cell. See README.md for methods and results.

In [ ]:
PROJECT_SOURCE = "\n# Credit risk project: complete English workflow.\n\nfrom pathlib import Path\nfrom zipfile import ZipFile, ZIP_DEFLATED\nimport json\nimport platform\nimport shutil\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom matplotlib.font_manager import FontProperties\nfrom matplotlib.ticker import PercentFormatter\nimport joblib\nimport nbformat\n\nfrom sklearn.base import BaseEstimator, TransformerMixin, clone\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.calibration import CalibratedClassifierCV, calibration_curve\nfrom sklearn.model_selection import (\n    train_test_split, StratifiedKFold, cross_val_score\n)\nfrom sklearn.metrics import (\n    roc_auc_score, roc_curve, confusion_matrix, brier_score_loss\n)\nfrom lightgbm import LGBMClassifier\n\nSEED = 42\nTARGET = \"SeriousDlqin2yrs\"\nLATE = [\n    \"NumberOfTime30-59DaysPastDueNotWorse\",\n    \"NumberOfTime60-89DaysPastDueNotWorse\",\n    \"NumberOfTimes90DaysLate\",\n]\nOUT = Path(\"/kaggle/working\")\nFIGURES = OUT / \"figures\"\nOUT.mkdir(parents=True, exist_ok=True)\nFIGURES.mkdir(parents=True, exist_ok=True)\n\ndef write_file(name, content):\n    path = OUT / name\n    path.write_text(content.strip() + \"\\n\", encoding=\"utf-8\")\n    return path\n\n# 1. Locate and validate competition data.\ntrain_files = list(Path(\"/kaggle/input\").rglob(\"cs-training.csv\"))\ntest_files = list(Path(\"/kaggle/input\").rglob(\"cs-test.csv\"))\nif len(train_files) != 1 or len(test_files) != 1:\n    raise RuntimeError(\n        \"Attach Give Me Some Credit through Add Input. \"\n        \"Exactly one cs-training.csv and one cs-test.csv are required.\"\n    )\n\nraw = pd.read_csv(train_files[0])\ntest_raw = pd.read_csv(test_files[0])\nrequired = [TARGET, \"age\", \"MonthlyIncome\", \"NumberOfDependents\",\n            \"DebtRatio\", \"RevolvingUtilizationOfUnsecuredLines\"] + LATE\nassert set(required).issubset(raw.columns), \"Required columns are missing.\"\nassert raw[TARGET].isin([0, 1]).all(), \"Invalid target labels.\"\n\nmissing_raw = raw.isna().sum().rename(\"missing_count\")\nmissing_raw.to_csv(OUT / \"raw_missing_values.csv\")\nraw.select_dtypes(\"number\").agg([\"min\", \"max\"]).T.to_csv(\n    OUT / \"raw_numeric_ranges.csv\"\n)\n\nprint(\"Raw training shape:\", raw.shape)\nprint(\"Raw test shape:\", test_raw.shape)\nprint(f\"Raw target rate: {raw[TARGET].mean():.2%}\")\n\n# 2. Clean records before splitting.\nclean = raw.drop(columns=[\"Unnamed: 0\"], errors=\"ignore\").copy()\ninvalid_age = int((clean[\"age\"] <= 0).sum())\nclean = clean.loc[clean[\"age\"] > 0].copy()\n\nabnormal = clean[LATE].isin([96, 98]).any(axis=1)\nabnormal_count = int(abnormal.sum())\nclean = clean.loc[~abnormal].copy()\n\nduplicate_count = int(clean.duplicated().sum())\nclean = clean.drop_duplicates().reset_index(drop=True)\n\nX = clean.drop(columns=TARGET)\ny = clean[TARGET].astype(int)\nX_train, X_valid, y_train, y_valid = train_test_split(\n    X, y, test_size=0.25, stratify=y, random_state=SEED\n)\noverall_rate = float(y_valid.mean())\n\n# Learn preprocessing values only from the fitting partition.\n# This transformer is also refitted inside each CV fold.\nclass CreditFeatures(BaseEstimator, TransformerMixin):\n    def fit(self, X, y=None):\n        self.columns_ = list(X.columns)\n        self.income_median_ = float(X[\"MonthlyIncome\"].median())\n        self.dependents_mode_ = float(\n            X[\"NumberOfDependents\"].mode().iloc[0]\n        )\n        self.debt_cap_ = float(X[\"DebtRatio\"].quantile(0.99))\n        return self\n\n    def transform(self, X):\n        d = X[self.columns_].copy()\n        d[\"MonthlyIncome_missing\"] = d[\"MonthlyIncome\"].isna().astype(int)\n        d[\"MonthlyIncome\"] = d[\"MonthlyIncome\"].fillna(self.income_median_)\n        d[\"NumberOfDependents\"] = d[\"NumberOfDependents\"].fillna(\n            self.dependents_mode_\n        )\n        d[\"ever_late\"] = (d[LATE] > 0).any(axis=1).astype(int)\n        d[\"max_late_level\"] = np.select(\n            [d[LATE[2]] > 0, d[LATE[1]] > 0, d[LATE[0]] > 0],\n            [3, 2, 1], default=0\n        )\n        d[\"log_MonthlyIncome\"] = np.log1p(\n            d[\"MonthlyIncome\"].clip(lower=0)\n        )\n        d[\"DebtRatio_capped\"] = d[\"DebtRatio\"].clip(\n            lower=0, upper=self.debt_cap_\n        )\n        d[\"log_DebtRatio\"] = np.log1p(d[\"DebtRatio_capped\"])\n        return d.astype(float)\n\nprep = CreditFeatures().fit(X_train)\ntraining_features = prep.transform(X_train)\nvalidation_features = prep.transform(X_valid)\nassert not training_features.isna().any().any()\nassert not validation_features.isna().any().any()\n\n# Save an analysis dataset using training-partition preprocessing values.\nanalysis_data = prep.transform(X)\nanalysis_data.insert(0, TARGET, y)\nanalysis_data.to_csv(OUT / \"cleaned_cs_training.csv\", index=False)\n\ncleaning_log = f\"\"\"\nCLEANING AND PREPROCESSING LOG\n\nRaw training rows: {len(raw):,}\nRaw training columns: {raw.shape[1]}\nRemoved the non-business row index column.\nInvalid-age rows removed: {invalid_age:,}\nAbnormal delinquency-code rows removed: {abnormal_count:,}\nExact duplicates removed before imputation: {duplicate_count:,}\nRetained borrower records: {len(clean):,}\n\nRaw MonthlyIncome missing values: {raw[\"MonthlyIncome\"].isna().sum():,}\nRaw NumberOfDependents missing values: {raw[\"NumberOfDependents\"].isna().sum():,}\nTraining income median: {prep.income_median_:,.2f}\nTraining dependents mode: {prep.dependents_mode_:g}\nTraining debt-ratio 99th-percentile cap: {prep.debt_cap_:,.4f}\n\nIncome missingness was retained as a binary feature.\nMissing income was filled with the fitting-partition median.\nMissing dependent counts were filled with the fitting-partition mode.\nImputation and feature transformations were refitted inside CV folds.\nTest records were retained; training-only row deletion was not applied to test data.\nDuplicates were identified before imputation to avoid treating a missing\nvalue as equivalent to an observed value that happens to equal the fill value.\n\nTraining rows: {len(X_train):,}\nValidation rows: {len(X_valid):,}\nTraining target rate: {y_train.mean():.4%}\nValidation target rate: {y_valid.mean():.4%}\nModel feature count: {training_features.shape[1]}\nRemaining missing model-feature values: 0\n\"\"\"\nwrite_file(\"cleaning_log.txt\", cleaning_log)\n\n# 3. Exploratory analysis.\ntarget_counts = y.value_counts().reindex([0, 1])\nlate_group = clean[LATE[2]].clip(upper=3).astype(int)\nlate_table = clean.groupby(late_group)[TARGET].agg(\n    count=\"size\", default_rate=\"mean\"\n).reindex([0, 1, 2, 3])\n\nutil_labels = [\"0-10%\", \"10-30%\", \"30-60%\", \"60-100%\", \">100%\"]\nutil_group = pd.cut(\n    clean[\"RevolvingUtilizationOfUnsecuredLines\"],\n    [-np.inf, 0.1, 0.3, 0.6, 1.0, np.inf],\n    labels=util_labels, include_lowest=True\n)\nutil_table = clean.groupby(util_group, observed=False)[TARGET].agg(\n    count=\"size\", default_rate=\"mean\"\n)\nlate_table.to_csv(OUT / \"late90_analysis.csv\")\nutil_table.to_csv(OUT / \"utilization_analysis.csv\")\n\ncorrelation = clean.select_dtypes(\"number\").corr()\ncorrelation.to_csv(OUT / \"correlation_matrix.csv\")\n\n# 4. Train models and evaluate five-fold CV on training data only.\nlr = Pipeline([\n    (\"features\", CreditFeatures()),\n    (\"scale\", StandardScaler()),\n    (\"model\", LogisticRegression(\n        max_iter=2000, class_weight=\"balanced\", random_state=SEED\n    )),\n])\nlgb = Pipeline([\n    (\"features\", CreditFeatures()),\n    (\"model\", LGBMClassifier(\n        n_estimators=300, learning_rate=0.05, num_leaves=31,\n        random_state=SEED, class_weight=None, verbosity=-1, n_jobs=2\n    )),\n])\n\nlr.fit(X_train, y_train)\nlgb.fit(X_train, y_train)\n\ncv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)\ncv_scores = cross_val_score(\n    clone(lgb), X_train, y_train, scoring=\"roc_auc\",\n    cv=cv, n_jobs=1\n)\n\n# Calibrate using training-only folds, keeping validation data separate.\nstrategy_model = CalibratedClassifierCV(\n    estimator=clone(lgb), method=\"sigmoid\",\n    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),\n    n_jobs=1\n)\nstrategy_model.fit(X_train, y_train)\n\nlr_prob = lr.predict_proba(X_valid)[:, 1]\nraw_prob = lgb.predict_proba(X_valid)[:, 1]\nprob = strategy_model.predict_proba(X_valid)[:, 1]\n\nmodel_comparison = pd.DataFrame([\n    {\n        \"model\": \"Logistic Regression\",\n        \"train_auc\": roc_auc_score(y_train, lr.predict_proba(X_train)[:, 1]),\n        \"validation_auc\": roc_auc_score(y_valid, lr_prob),\n        \"cv_mean_auc\": np.nan, \"cv_std_auc\": np.nan,\n    },\n    {\n        \"model\": \"LightGBM\",\n        \"train_auc\": roc_auc_score(y_train, lgb.predict_proba(X_train)[:, 1]),\n        \"validation_auc\": roc_auc_score(y_valid, raw_prob),\n        \"cv_mean_auc\": cv_scores.mean(), \"cv_std_auc\": cv_scores.std(),\n    },\n    {\n        \"model\": \"Calibrated LightGBM\",\n        \"train_auc\": roc_auc_score(\n            y_train, strategy_model.predict_proba(X_train)[:, 1]\n        ),\n        \"validation_auc\": roc_auc_score(y_valid, prob),\n        \"cv_mean_auc\": np.nan, \"cv_std_auc\": np.nan,\n    },\n])\nmodel_comparison.to_csv(OUT / \"model_comparison.csv\", index=False)\n\nauc = float(roc_auc_score(y_valid, prob))\nfpr, tpr, roc_thresholds = roc_curve(y_valid, prob)\nks_index = int(np.argmax(tpr - fpr))\nks = float((tpr - fpr)[ks_index])\nks_threshold = float(roc_thresholds[ks_index])\nbrier = float(brier_score_loss(y_valid, prob))\ncm = confusion_matrix(y_valid, (prob >= 0.10).astype(int))\npd.DataFrame(cm, index=[\"Actual 0\", \"Actual 1\"],\n             columns=[\"Predicted 0\", \"Predicted 1\"]).to_csv(\n    OUT / \"confusion_matrix.csv\"\n)\n\nevaluation = pd.DataFrame({\"actual\": y_valid.to_numpy(), \"probability\": prob})\nevaluation[\"decile\"] = (\n    pd.qcut(evaluation[\"probability\"].rank(method=\"first\", ascending=False),\n            10, labels=False) + 1\n)\nlift_table = evaluation.groupby(\"decile\")[\"actual\"].agg(\n    count=\"size\", default_rate=\"mean\"\n)\nlift_table[\"lift\"] = lift_table[\"default_rate\"] / overall_rate\nlift_table.to_csv(OUT / \"lift_table.csv\")\ntop_rate = float(lift_table.loc[1, \"default_rate\"])\ntop_lift = float(lift_table.loc[1, \"lift\"])\n\nimportance = pd.DataFrame({\n    \"feature\": training_features.columns,\n    \"importance\": lgb.named_steps[\"model\"].feature_importances_,\n}).sort_values(\"importance\", ascending=False)\nimportance.to_csv(OUT / \"feature_importance.csv\", index=False)\n\n# 5. Risk segmentation and approval simulations.\nevaluation[\"risk_group\"] = pd.cut(\n    prob, [-np.inf, 0.05, 0.15, np.inf], right=False,\n    labels=[\"Low Risk (<5%)\", \"Medium Risk (5%-15%)\", \"High Risk (>=15%)\"]\n)\nrisk_table = evaluation.groupby(\"risk_group\", observed=False)[\"actual\"].agg(\n    count=\"size\", actual_default_rate=\"mean\"\n)\nrisk_table[\"group_share\"] = risk_table[\"count\"] / len(evaluation)\nrisk_table.to_csv(OUT / \"risk_segmentation.csv\")\n\nGOOD_PROFIT = 1000\nBAD_LOSS = 7000\nbreak_even = GOOD_PROFIT / (GOOD_PROFIT + BAD_LOSS)\nrows = []\n\nfor threshold in [0.05, 0.10, 0.15, 0.20]:\n    approved = prob < threshold\n    n = int(approved.sum())\n    bad = int(evaluation.loc[approved, \"actual\"].sum())\n    good = n - bad\n    retrospective_profit = good * GOOD_PROFIT - bad * BAD_LOSS\n    probability_profit = float(\n        (GOOD_PROFIT * (1 - prob[approved]) - BAD_LOSS * prob[approved]).sum()\n    )\n    rows.append({\n        \"approval_threshold\": threshold,\n        \"approved_count\": n,\n        \"rejected_count\": len(prob) - n,\n        \"approval_rate\": n / len(prob),\n        \"observed_default_rate\": bad / n if n else np.nan,\n        \"good_count\": good,\n        \"default_count\": bad,\n        \"simulated_profit\": retrospective_profit,\n        \"probability_based_expected_profit\": probability_profit,\n        \"average_profit_per_approved\": retrospective_profit / n if n else np.nan,\n    })\n\nstrategy = pd.DataFrame(rows)\nstrategy.to_csv(OUT / \"approval_profit_simulation.csv\", index=False)\nbest = strategy.loc[strategy[\"simulated_profit\"].idxmax()]\nbest_threshold = float(best[\"approval_threshold\"])\nbest_profit = float(best[\"simulated_profit\"])\n\n# 6. Save figures.\nplt.rcParams.update({\n    \"font.family\": \"DejaVu Sans\", \"pdf.fonttype\": 42,\n    \"axes.spines.top\": False, \"axes.spines.right\": False,\n})\nBLUE = \"#2878A8\"\nNAVY = \"#17324D\"\n\nshort_names = {\n    \"RevolvingUtilizationOfUnsecuredLines\": \"Credit utilization\",\n    \"age\": \"Age\", \"MonthlyIncome\": \"Monthly income\",\n    \"DebtRatio\": \"Debt ratio\",\n    \"NumberOfOpenCreditLinesAndLoans\": \"Open credit lines\",\n    \"DebtRatio_capped\": \"Capped debt ratio\",\n    \"NumberRealEstateLoansOrLines\": \"Real estate loans\",\n    LATE[0]: \"30-59 days late\", LATE[1]: \"60-89 days late\",\n    LATE[2]: \"90+ days late\",\n    \"NumberOfDependents\": \"Dependents\",\n    \"MonthlyIncome_missing\": \"Income missing\",\n    \"ever_late\": \"Any past delinquency\",\n    \"max_late_level\": \"Maximum late level\",\n    \"log_MonthlyIncome\": \"Log income\", \"log_DebtRatio\": \"Log debt ratio\",\n}\ntop_importance = importance.head(10)\n\ndef draw_chart(ax, kind):\n    if kind == \"target\":\n        ax.bar([\"No serious\\ndelinquency\", \"Serious\\ndelinquency\"],\n               target_counts.to_numpy(), color=[BLUE, NAVY])\n        ax.set_title(\"Target Distribution\")\n        ax.set_ylabel(\"Borrowers\")\n        ax.ticklabel_format(axis=\"y\", style=\"plain\")\n    elif kind in [\"late\", \"util\"]:\n        table = late_table if kind == \"late\" else util_table\n        labels = [\"0\", \"1\", \"2\", \"3+\"] if kind == \"late\" else util_labels\n        values = table[\"default_rate\"].to_numpy()\n        bars = ax.bar(labels, values, color=BLUE)\n        ax.set_title(\"Past 90+ Days Late and Risk\" if kind == \"late\"\n                     else \"Credit Utilization and Risk\")\n        ax.set_xlabel(\"Past 90+ day delinquencies\" if kind == \"late\"\n                      else \"Credit utilization\")\n        ax.set_ylabel(\"Observed target rate\")\n        ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))\n        ax.set_ylim(0, np.nanmax(values) * 1.18)\n        for bar, value in zip(bars, values):\n            ax.text(bar.get_x() + bar.get_width()/2, value + 0.008,\n                    f\"{value:.1%}\", ha=\"center\", fontsize=7)\n    else:\n        names = top_importance[\"feature\"].map(short_names).fillna(\n            top_importance[\"feature\"]\n        )\n        ax.barh(names, top_importance[\"importance\"], color=BLUE)\n        ax.invert_yaxis()\n        ax.set_title(\"Feature Importance\")\n        ax.set_xlabel(\"Split count\")\n    ax.set_axisbelow(True)\n    ax.grid(axis=\"x\" if kind == \"importance\" else \"y\",\n            color=\"#E8EDF2\", linewidth=0.6)\n    ax.tick_params(labelsize=7, length=0)\n    ax.title.set_fontsize(9)\n    ax.xaxis.label.set_fontsize(8)\n    ax.yaxis.label.set_fontsize(8)\n\nchart_files = {\n    \"target\": \"target_distribution.png\",\n    \"late\": \"late90_default_rate.png\",\n    \"util\": \"credit_utilization_default_rate.png\",\n    \"importance\": \"feature_importance_top10.png\",\n}\nfor kind, name in chart_files.items():\n    fig, ax = plt.subplots(figsize=(7, 4))\n    draw_chart(ax, kind)\n    fig.tight_layout()\n    fig.savefig(FIGURES / name, dpi=220, bbox_inches=\"tight\")\n    plt.close(fig)\n\nfig, ax = plt.subplots(figsize=(6, 5))\nax.plot(fpr, tpr, color=BLUE, label=f\"AUC = {auc:.4f}\")\nax.plot([0, 1], [0, 1], \"--\", color=\"gray\")\nax.set(xlabel=\"False positive rate\", ylabel=\"True positive rate\",\n       title=\"Validation ROC Curve\")\nax.legend()\nfig.tight_layout()\nfig.savefig(FIGURES / \"roc_curve.png\", dpi=220)\nplt.close(fig)\n\nobserved, predicted = calibration_curve(y_valid, prob, n_bins=10,\n                                        strategy=\"quantile\")\nfig, ax = plt.subplots(figsize=(6, 5))\nax.plot(predicted, observed, \"o-\", color=BLUE)\nax.plot([0, 1], [0, 1], \"--\", color=\"gray\")\nax.set(xlabel=\"Mean predicted probability\", ylabel=\"Observed target rate\",\n       title=\"Validation Calibration\")\nfig.tight_layout()\nfig.savefig(FIGURES / \"calibration_curve.png\", dpi=220)\nplt.close(fig)\n\n# 7. Generate the one-page PDF with justified paragraphs and four charts.\nsections = [\n    (\"Project Background\",\n     \"Predict serious delinquency within two years using Kaggle borrower \"\n     \"records, and translate predicted probabilities into simulated \"\n     \"credit approval decisions.\"),\n    (\"Data and Cleaning\",\n     f\"The raw dataset contained {len(raw):,} records and {raw.shape[1]} columns. \"\n     f\"Cleaning removed the row index, {invalid_age:,} invalid-age record(s), \"\n     f\"{abnormal_count:,} abnormal-code records and {duplicate_count:,} duplicates. \"\n     f\"{len(clean):,} records remained. Income missingness was retained; \"\n     \"imputation and debt capping were learned from the fitting partition.\"),\n    (\"Methods and Metrics\",\n     \"A stratified 75%/25% split was used. Logistic Regression and LightGBM \"\n     f\"were compared. Calibrated LightGBM validation AUC was {auc:.4f}, \"\n     f\"KS was {ks:.4f}, and Brier score was {brier:.4f}. \"\n     f\"Five-fold training-set LightGBM CV AUC was {cv_scores.mean():.4f} \"\n     f\"(standard deviation {cv_scores.std():.4f}). \"\n     \"Sigmoid calibration used training-only folds.\"),\n    (\"Strategy Conclusion\",\n     f\"The highest-risk 10% had a target rate of {top_rate:.2%}, versus \"\n     f\"{overall_rate:.2%} overall (lift {top_lift:.2f}). Assuming RMB 1,000 \"\n     \"profit per good loan and RMB 7,000 loss per bad loan, \"\n     f\"the {best_threshold:.0%} cutoff gave the highest retrospective simulated \"\n     f\"profit among four tested cutoffs: RMB {best_profit:,.0f}, with \"\n     f\"{best['approval_rate']:.2%} approved and \"\n     f\"{best['observed_default_rate']:.2%} observed delinquency among approved \"\n     \"borrowers. This validation-set result requires independent confirmation.\"),\n    (\"Tools\",\n     \"Python, pandas, NumPy, matplotlib, scikit-learn, LightGBM, \"\n     \"joblib, and Kaggle Notebook.\"),\n]\n\nfig = plt.figure(figsize=(8.27, 11.69), facecolor=\"white\")\nfig.text(0.075, 0.955, \"Credit Risk Prediction\",\n         fontsize=20, weight=\"bold\", color=NAVY)\nfig.text(0.075, 0.931,\n         \"Give Me Some Credit | One-Page Project Summary\",\n         fontsize=10, color=\"#52606D\")\n\nfig.canvas.draw()\nrenderer = fig.canvas.get_renderer()\npage_width = fig.bbox.width\nfont = FontProperties(family=\"DejaVu Sans\", size=8.5)\navailable_width = 0.85 * page_width\n\ndef text_width(text):\n    return renderer.get_text_width_height_descent(text, font, False)[0]\n\ndef add_justified_section(y_pos, heading, body):\n    fig.text(0.075, y_pos, heading, fontsize=9.5,\n             weight=\"bold\", color=NAVY, va=\"top\")\n    lines, current = [], []\n    for word in body.split():\n        if current and text_width(\" \".join(current + [word])) > available_width:\n            lines.append(current)\n            current = [word]\n        else:\n            current.append(word)\n    if current:\n        lines.append(current)\n    step = (8.5 * 1.35 / 72) / fig.get_figheight()\n    for i, words in enumerate(lines):\n        line_y = y_pos - 0.019 - i * step\n        if i == len(lines) - 1 or len(words) == 1:\n            fig.text(0.075, line_y, \" \".join(words),\n                     fontproperties=font, va=\"top\", color=\"#222222\")\n        else:\n            widths = [text_width(w) for w in words]\n            gap = (available_width - sum(widths)) / (len(words) - 1)\n            x_pos = 0.075\n            for word, width in zip(words, widths):\n                fig.text(x_pos, line_y, word,\n                         fontproperties=font, va=\"top\", color=\"#222222\")\n                x_pos += (width + gap) / page_width\n    return y_pos - 0.019 - len(lines) * step - 0.020\n\ncursor = 0.895\nfor heading, body in sections:\n    cursor = add_justified_section(cursor, heading, body)\n\nif cursor < 0.47:\n    raise RuntimeError(\"Summary text exceeds the reserved text area.\")\n\npositions = [\n    [0.115, 0.270, 0.345, 0.165],\n    [0.585, 0.270, 0.345, 0.165],\n    [0.115, 0.060, 0.345, 0.165],\n    [0.705, 0.060, 0.225, 0.165],\n]\nfor kind, position in zip(chart_files, positions):\n    ax = fig.add_axes(position)\n    draw_chart(ax, kind)\nfig.savefig(OUT / \"one_page_project_summary.pdf\", format=\"pdf\")\nplt.show()\nplt.close(fig)\n\n# 8. Refit the submission model on all retained training records.\nfinal_model = clone(strategy_model).fit(X, y)\njoblib.dump(final_model, OUT / \"lightgbm_credit_model.pkl\")\njoblib.dump(lr, OUT / \"logistic_regression_model.pkl\")\n\nassert \"Unnamed: 0\" in test_raw.columns, \"Test IDs are missing.\"\ntest_ids = test_raw[\"Unnamed: 0\"].copy()\nX_test = test_raw.drop(columns=[\"Unnamed: 0\", TARGET], errors=\"ignore\")\ntest_prob = final_model.predict_proba(X_test[X.columns])[:, 1]\nsubmission = pd.DataFrame({\"Id\": test_ids, \"Probability\": test_prob})\nassert len(submission) == len(test_raw)\nassert submission[\"Id\"].is_unique\nassert submission[\"Probability\"].between(0, 1).all()\nassert not submission.isna().any().any()\nsubmission.to_csv(OUT / \"submission.csv\", index=False)\n\n# 9. Write English documentation with actual results.\nsummary_md = \"# One-Page Project Summary\\n\\n\" + \"\\n\\n\".join(\n    f\"## {heading}\\n\\n{body}\" for heading, body in sections\n)\nwrite_file(\"final_project_summary.md\", summary_md)\n\nrecommendations = \"\"\"\n1. Review borrowers with severe past delinquency and high utilization\n   more strictly. Validate any pricing or limit changes separately.\n2. Consider reduced limits, manual review or additional guarantees for\n   medium-risk borrowers rather than automatic rejection.\n3. Monitor approval rate, observed target rate, AUC, KS, calibration and\n   population stability. Outcome-based metrics require matured labels.\n\"\"\"\nlimitations = \"\"\"\nThe target is serious delinquency, not observed financial loss.\nLoan amount and profit/loss assumptions are illustrative.\nThe recommended cutoff is the best of four tested cutoffs on this\nvalidation set; it is not a proven global optimum.\nThreshold selection uses validation labels and must be confirmed on\nindependent data before deployment.\nProbability calibration should be reassessed over time.\nHistorical borrower data may not represent the population of applicants\nwho would be approved or rejected by a new lending policy.\nFeature split counts measure model usage, not causal effects.\n\"\"\"\n\nwrite_file(\"project_report.md\", f\"\"\"\n# Credit Risk Prediction Project Report\n\n{summary_md.replace(\"# One-Page Project Summary\", \"\")}\n\n## Exploratory Analysis\nThe retained-data target rate was {y.mean():.2%}.\nPredicting every borrower as non-delinquent would yield apparent\naccuracy of {1-y.mean():.2%}; accuracy alone is therefore inadequate.\nSee the delinquency and utilization tables, correlation matrix and figures.\n\n## Evaluation\nValidation AUC: {auc:.4f}\nValidation KS: {ks:.4f}\nMaximum-KS score threshold: {ks_threshold:.4f}\nBrier score: {brier:.4f}\nHighest-risk-decile lift: {top_lift:.2f}\n\n## Approval Strategy\nThe theoretical break-even probability is {break_even:.2%}.\nThe best retrospectively simulated cutoff among the tested choices is\n{best_threshold:.0%}. Simulated profit is RMB {best_profit:,.0f}.\nThe strategy table also reports probability-based expected profit.\nThe two profit measures should not be treated as identical.\n\n## Business Recommendations\n{recommendations}\n\n## Limitations\n{limitations}\n\"\"\")\n\nwrite_file(\"DATA_DESCRIPTION.md\", f\"\"\"\n# Data Description\n\n## Source\nKaggle Give Me Some Credit:\nhttps://www.kaggle.com/competitions/GiveMeSomeCredit\n\n## Records and Target\nRaw training records: {len(raw):,}\nTest records: {len(test_raw):,}\nRetained training records: {len(clean):,}\nSeriousDlqin2yrs equals 1 for serious delinquency within two years\nand 0 otherwise. The test target labels are not supplied.\n\n## Access and Use\nObtain the original competition data directly from Kaggle and follow\nthe applicable competition rules and data-use terms.\nRaw and cleaned borrower data are excluded from the portfolio ZIP.\nThe Kaggle working directory contains a cleaned analysis CSV separately.\n\n## Interpretation\nThe target is serious delinquency, not realized loan loss.\nProfit and loss assumptions are illustrative.\nSee cleaning_log.txt and project_report.md for processing and limitations.\n\"\"\")\n\nwrite_file(\"README.md\", f\"\"\"\n# Credit Risk Prediction - Give Me Some Credit\n\nThis student project predicts serious delinquency within two years and\ncompares credit approval strategies under illustrative financial assumptions.\n\n## Data\nSource: https://www.kaggle.com/competitions/GiveMeSomeCredit\nSee DATA_DESCRIPTION.md. Obtain data directly from Kaggle.\n\n## Run\nOpen analysis.ipynb in Kaggle, attach Give Me Some Credit through Add Input,\nand choose Run All. No original borrower CSV files are bundled.\nThe code locates cs-training.csv and cs-test.csv under /kaggle/input.\nGenerated outputs are saved under /kaggle/working.\nInstall packages listed in requirements.txt if using a compatible environment.\n\n## Workflow\nRecord cleaning; training-only preprocessing; exploratory analysis;\nLogistic Regression; LightGBM; five-fold CV; training-only sigmoid\ncalibration; validation AUC, KS, lift, confusion matrix and calibration;\nrisk segmentation; approval and profit simulation; final full-data refit.\n\n## Results\nCalibrated LightGBM validation AUC: {auc:.4f}\nValidation KS: {ks:.4f}\nTraining-set LightGBM five-fold CV AUC: {cv_scores.mean():.4f}\nCV standard deviation: {cv_scores.std():.4f}\nTop-decile lift: {top_lift:.2f}\nBest retrospectively simulated cutoff among four tested options: {best_threshold:.0%}\nSimulated validation profit: RMB {best_profit:,.0f}\n\n## Main Deliverables\n- analysis.ipynb: complete runnable notebook\n- credit_risk_project.py: the same workflow as a script\n- cleaning_log.txt: processing counts and decisions\n- one_page_project_summary.pdf: one-page summary with four charts\n- project_report.md: extended report and limitations\n- DATA_DESCRIPTION.md: data source and usage notes\n- figures/: analysis and evaluation figures\n- model_comparison.csv: model results\n- approval_profit_simulation.csv: strategy comparisons\n- resume_bullet.txt: project description for a resume\n- requirements.txt: installed package versions\n\n## Limitations\n{limitations}\n\"\"\")\n\nwrite_file(\"resume_bullet.txt\", f\"\"\"\nDeveloped a Python credit risk model using {len(raw):,} Kaggle\nborrower records; completed record cleaning, feature engineering,\nLogistic Regression and LightGBM comparisons, and training-only probability\ncalibration. Achieved validation AUC {auc:.4f} and KS {ks:.4f}, and translated\npredictions into approval and profit simulations under stated assumptions.\n\"\"\")\n\nimport importlib.metadata as metadata\npackages = [\"numpy\", \"pandas\", \"matplotlib\", \"scikit-learn\",\n            \"lightgbm\", \"joblib\", \"nbformat\"]\nwrite_file(\"requirements.txt\", \"\\n\".join(\n    f\"{package}=={metadata.version(package)}\" for package in packages\n))\nwrite_file(\"run_environment.json\", json.dumps({\n    \"python\": platform.python_version(),\n    \"seed\": SEED,\n    \"train_path\": str(train_files[0]),\n    \"test_path\": str(test_files[0]),\n    \"cv_scores\": cv_scores.tolist(),\n}, indent=2))\n\n# 10. Export a runnable English notebook and script.\n# The source wrapper keeps notebook exports reproducible.\nrunnable_source = (\n    \"PROJECT_SOURCE = \" + repr(PROJECT_SOURCE) +\n    \"\\nexec(compile(PROJECT_SOURCE, 'credit_risk_project.py', 'exec'))\\n\"\n)\nwrite_file(\"credit_risk_project.py\", runnable_source)\nnotebook = nbformat.v4.new_notebook(\n    cells=[\n        nbformat.v4.new_markdown_cell(\n            \"# Credit Risk Prediction - Give Me Some Credit\\n\\n\"\n            \"Complete English analysis and deliverable generation. \"\n            \"Attach the competition data in Kaggle and run the code cell.\"\n        ),\n        nbformat.v4.new_code_cell(runnable_source),\n    ],\n    metadata={\n        \"kernelspec\": {\n            \"display_name\": \"Python 3\",\n            \"language\": \"python\",\n            \"name\": \"python3\"\n        }\n    }\n)\nnbformat.write(notebook, OUT / \"analysis.ipynb\")\n\n# Package public portfolio files without borrower-level data.\nportfolio_names = [\n    \"analysis.ipynb\", \"credit_risk_project.py\", \"README.md\",\n    \"DATA_DESCRIPTION.md\", \"cleaning_log.txt\",\n    \"one_page_project_summary.pdf\", \"final_project_summary.md\",\n    \"project_report.md\", \"resume_bullet.txt\", \"requirements.txt\",\n    \"run_environment.json\", \"model_comparison.csv\",\n    \"approval_profit_simulation.csv\", \"risk_segmentation.csv\",\n    \"lift_table.csv\", \"confusion_matrix.csv\", \"feature_importance.csv\",\n    \"raw_missing_values.csv\", \"raw_numeric_ranges.csv\",\n    \"late90_analysis.csv\", \"utilization_analysis.csv\",\n    \"correlation_matrix.csv\",\n]\nzip_path = OUT / \"credit_risk_portfolio.zip\"\nwith ZipFile(zip_path, \"w\", ZIP_DEFLATED) as archive:\n    for name in portfolio_names:\n        archive.write(OUT / name, arcname=name)\n    for path in sorted(FIGURES.glob(\"*.png\")):\n        archive.write(path, arcname=f\"figures/{path.name}\")\n\nwith ZipFile(zip_path) as archive:\n    assert archive.testzip() is None, \"ZIP integrity check failed.\"\n\n# Verify that all required generated files exist and are nonempty.\nchecks = portfolio_names + [\n    \"submission.csv\", \"lightgbm_credit_model.pkl\",\n    \"logistic_regression_model.pkl\", \"cleaned_cs_training.csv\",\n    \"credit_risk_portfolio.zip\",\n]\nstatus = pd.DataFrame([\n    {\"file\": name, \"exists\": (OUT / name).is_file(),\n     \"bytes\": (OUT / name).stat().st_size if (OUT / name).is_file() else 0}\n    for name in checks\n])\nassert status[\"exists\"].all() and (status[\"bytes\"] > 0).all()\nstatus.to_csv(OUT / \"deliverable_check.csv\", index=False)\n\nprint(\"\\nMODEL COMPARISON\")\nprint(model_comparison.round(4).to_string(index=False))\nprint(\"\\nRISK SEGMENTATION\")\nprint(risk_table.to_string())\nprint(\"\\nAPPROVAL AND PROFIT SIMULATION\")\nprint(strategy.round(4).to_string(index=False))\nprint(\"\\nDELIVERABLE CHECK\")\nprint(status.to_string(index=False))\nprint(\"\\nSubmission shape:\", submission.shape)\nprint(\"One-page PDF:\", OUT / \"one_page_project_summary.pdf\")\nprint(\"GitHub portfolio ZIP:\", zip_path)\nprint(\"\\nALL GENERATED FILE CHECKS PASSED.\")\nprint(\"Save a successful committed version before downloading final outputs.\")\nprint(\"Competition submission and GitHub publishing require separate actions.\")\n"
exec(compile(PROJECT_SOURCE, 'credit_risk_project.py', 'exec'))
